# Financial Market Intelligence System — reproducing the run

**SDAIA Academy · Modern Data Engineering for AI Systems**

> **The captured evidence for this project lives in [`evidence/`](../evidence/), not in this
> notebook.** That directory holds the committed output of a real end-to-end run: the data
> contract scorecard, the refused writes, the quality gate results, the Airflow task states,
> the OpenLineage events and the cited RAG answers. Start there — it is the submission.
>
> **This notebook is the tool for reproducing that run**, stage by stage, with each step's
> artefact inspected inline as it is produced. It is deliberately committed without stored
> output so it always reflects the code rather than a stale execution.

To run it, bring the infrastructure up first:

```bash
make infra-up
make ollama-pull
```

then execute top to bottom with the `.venv-app` kernel. Budget roughly an hour on a CPU-only
machine — embedding ~5,800 chunks and generating six answers on a local 7B model dominate
that time. See the performance table in the [README](../README.md).

| Section | Rubric deliverable | Committed evidence |
| --- | --- | --- |
| 1 | Ingestion — Kafka + data contract + dead-letter routing (20) | `evidence/runs/contract_scorecard.md` |
| 2 | Delta Lakehouse — Bronze/Silver/Gold + MERGE + schema enforcement (25) | `evidence/runs/gold_table.md`, `schema_enforcement.json` |
| 3 | RAG — chunking, hybrid search, RRF, reranking, cited answers (25) | `evidence/runs/rag_answers.md` |
| 4 | Quality gate + lineage — Great Expectations and OpenLineage (15) | `evidence/runs/quality_gate*.json`, `evidence/lineage/` |
| 5 | Orchestration — the Airflow DAG and its failure path (15) | `evidence/runs/airflow_gate_failure_dag.md` |

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

import pandas as pd

from fmis.config import REPO_ROOT, settings


def run(*args: str) -> int:
    """Invoke a pipeline stage exactly as Airflow does, streaming its output."""
    cmd = [sys.executable, "-m", "fmis.cli", *args]
    print(f"$ fmis {' '.join(args)}\n")
    result = subprocess.run(cmd, cwd=REPO_ROOT, text=True)
    print(f"\n[exit code: {result.returncode}]")
    return result.returncode


def evidence(name: str) -> dict:
    return json.loads((settings.evidence_root / "runs" / name).read_text())


print(f"repo root : {REPO_ROOT}")
print(f"lakehouse : {settings.lakehouse_root}")
print(f"tickers   : {settings.tickers}")

---
## 1 · Ingestion — the data contract at the boundary

The producer streams real daily quotes to a real Kafka broker and deliberately corrupts a
configurable slice of them. The consumer applies the `StockQuote` Pydantic contract to
every message and routes failures to **both** a dead-letter topic and an on-disk quarantine
zone, each carrying the reason code and the field-level violations.

The corruption is tagged in a Kafka header **for scoring only** — the consumer never reads
it, and decides purely from the message body.

In [ ]:
from fmis.ingestion.corruption import CORRUPTIONS

pd.DataFrame(
    [
        {
            "fault": c.name,
            "what it does": c.description,
            "expected reason code": c.expected_reason.value,
        }
        for c in CORRUPTIONS
    ]
)

In [ ]:
run("kafka-setup")
run("produce")
run("consume")

In [ ]:
# Every injected fault, and whether the contract caught it.
run("contract-report")

scorecard = evidence("contract_scorecard.json")
print(f"\nContract catch rate : {scorecard['contract_catch_rate']}")
print(f"Faults that escaped : {scorecard['faults_escaped_total']}")
pd.DataFrame(scorecard["per_fault"])

In [ ]:
# Three quarantined records, showing the recorded rejection reason.
quarantine = sorted(settings.quarantine_path.glob("rejected_*.jsonl"))[-1]

with quarantine.open() as handle:
    for line, _ in zip(handle, range(3)):
        record = json.loads(line)
        print(f"reason     : {record['reason']}")
        print(f"violation  : {record['violations'][0]['message']}")
        print(f"received   : {record['violations'][0]['received']}")
        print(f"raw payload: {record['raw_payload'][:120]}")
        print(f"offset     : {record['kafka']['topic']}@{record['kafka']['offset']}")
        print("-" * 78)

In [ ]:
# Independent confirmation that messages really traversed a broker.
from fmis.ingestion.kafka_admin import topic_counts

topic_counts()

---
## 2 · Delta Lakehouse — Bronze, Silver, Gold

Bronze keeps the raw JSON verbatim. Silver parses it against an explicit schema, casts every
column, deduplicates on `(ticker, trade_date)` and carries Delta `CHECK` constraints. Gold is
a **genuine aggregate** — one row per ticker, derived from windows over the whole history —
maintained by a real `MERGE` on the ticker business key.

In [ ]:
run("bronze")
run("silver")

In [ ]:
from fmis.lakehouse.session import get_spark

spark = get_spark("notebook")

bronze = spark.read.format("delta").load(str(settings.bronze_path))
silver = spark.read.format("delta").load(str(settings.silver_path))

print("BRONZE — raw JSON preserved verbatim, with ingestion provenance")
bronze.select("raw_payload", "ingest_ts", "batch_id").show(2, truncate=90)

print("SILVER — parsed and correctly typed")
silver.printSchema()
silver.orderBy("ticker", "trade_date").show(5)

In [ ]:
# Storage-level constraints: enforced for ANY writer, not just this pipeline.
spark.sql(f"SHOW TBLPROPERTIES delta.`{settings.silver_path}`").filter(
    "key LIKE 'delta.constraints%'"
).show(truncate=False)

### 2a · Schema enforcement — bad writes are refused

Six deliberately invalid writes, exercising two distinct mechanisms: Delta's schema
enforcement (schema evolution is off) and its stored `NOT NULL` / `CHECK` invariants. A
well-formed control row is written first, so a "refused" result proves enforcement rather
than a broken table.

In [ ]:
run("enforce-schema")

enforcement = evidence("schema_enforcement.json")
print(f"enforcement holds: {enforcement['enforcement_holds']}")
pd.DataFrame(enforcement["attempts"])[
    ["attempt", "mechanism", "expected", "refused", "exception_type"]
]

In [ ]:
# The actual error Delta raised for each refused write.
for attempt in enforcement["attempts"]:
    if attempt["refused"]:
        print(f"{attempt['attempt']}:\n  {attempt['message'][:220]}\n")

---
## 4 · Quality gate — Great Expectations, before Gold

The gate runs between Silver and Gold. The rubric's named check (`high >= low`) leads the
suite; the rest close the surrounding gaps.

In [ ]:
from fmis.quality.suite import build_specs

pd.DataFrame(
    [
        {"expectation": s.expectation, "parameters": s.kwargs, "why": s.rationale}
        for s in build_specs(settings.tickers)
    ]
)

In [ ]:
gate_exit = run("quality-gate")
assert gate_exit == 0, "gate should pass on clean Silver data"

gate = evidence("quality_gate.json")
print(f"success   : {gate['success']}")
print(f"evaluated : {gate['successful']}/{gate['evaluated']} expectations passed")
print(f"rows      : {gate['rows_validated']}")

### 4a · The gate actually bites

The same suite, run against Silver with violations injected **in memory only** — an inverted
high/low, a null close, a duplicated business key. Nothing is written to Delta.

**A non-zero exit code here is the deliverable.** It is what causes Airflow to fail the task
and skip everything downstream.

In [ ]:
failed_exit = run("quality-gate", "--taint")
assert failed_exit != 0, "the tainted gate MUST fail — that is the point"

demo = evidence("quality_gate_failure_demo.json")
print(f"\nsuccess: {demo['success']}  (expected: False)\n")
for failure in demo["failed_expectations"]:
    print(f"FAILED  {failure['expectation']}")
    print(f"        kwargs           : {failure['kwargs']}")
    print(f"        unexpected count : {failure['unexpected_count']}")

In [ ]:
# Confirm nothing was persisted: Silver is unchanged and still passes.
print(f"Silver row count after the tainted run: {silver.count()}")
assert run("quality-gate") == 0, "Silver must still be clean"

### 2b · Gold — a real MERGE on the ticker business key

Only now, past the gate, does Gold get written.

In [ ]:
run("gold")

gold = spark.read.format("delta").load(str(settings.gold_path))
gold.orderBy("ticker").select(
    "ticker",
    "as_of_date",
    "latest_close",
    "ma_30",
    "ma_30_trend",
    "pct_vs_ma_30",
    "volatility_30d_annualised",
    "pct_from_52w_high",
    "sessions_observed",
).show(truncate=False)

In [ ]:
# Proof it is an upsert, not an overwrite: re-running updates rather than inserts.
from fmis.lakehouse.session import delta_history

run("gold")

for entry in delta_history(settings.gold_path, limit=2):
    metrics = entry["operationMetrics"]
    print(
        f"v{entry['version']}  {entry['operation']:8}  "
        f"inserted={metrics.get('numTargetRowsInserted')}  "
        f"updated={metrics.get('numTargetRowsUpdated')}"
    )

print("\nFirst run inserts every ticker; the second updates them all.")
print("Gold is one row per ticker regardless of how many times it runs:")
print(f"  gold rows = {gold.count()}, distinct tickers = {gold.select('ticker').distinct().count()}")

In [ ]:
# Gold is an aggregate, not a copy: the grain differs by orders of magnitude.
print(f"Silver rows (ticker x session) : {silver.count():,}")
print(f"Gold rows   (ticker)           : {gold.count():,}")

---
## 3 · RAG — hybrid retrieval over 10-K filings

Section-aware chunking, a Chroma dense index and a BM25 sparse index over the same chunks,
fused with Reciprocal Rank Fusion, then reranked by a cross-encoder before the passages
reach the model.

In [ ]:
run("rag-index")

index_stats = evidence("rag_index.json")
print(json.dumps(index_stats, indent=2)[:900])

In [ ]:
# Each retriever alone, then the fusion — showing they disagree, which is the point.
from fmis.rag.retrieve import dense_search, reciprocal_rank_fusion, sparse_search

QUESTION = "What supply chain risks and component shortages threaten operations?"

dense_hits = dense_search(QUESTION)
sparse_hits = sparse_search(QUESTION)

print("DENSE top 5")
for hit in dense_hits[:5]:
    print(f"  {hit.dense_rank}. {hit.citation_label}  (cos={hit.dense_score})")

print("\nBM25 top 5")
for hit in sparse_hits[:5]:
    print(f"  {hit.sparse_rank}. {hit.citation_label}  (bm25={hit.sparse_score})")

overlap = {h.chunk_id for h in dense_hits[:5]} & {h.chunk_id for h in sparse_hits[:5]}
print(f"\nShared between the two top-5 lists: {len(overlap)} of 5")

In [ ]:
# RRF fusion, then cross-encoder reranking. The last column shows the reranker moving things.
from fmis.rag.rerank import rerank

fused = reciprocal_rank_fusion([dense_hits, sparse_hits])
rrf_order = [h.chunk_id for h in fused[: settings.rag_rerank_top_n]]
selected = rerank(QUESTION, fused)

pd.DataFrame(
    [
        {
            "final": h.final_rank,
            "source": h.citation_label,
            "found by": "+".join(h.retrievers),
            "dense rank": h.dense_rank,
            "bm25 rank": h.sparse_rank,
            "rrf": round(h.rrf_score, 5),
            "rerank": round(h.rerank_score, 3),
            "promoted by reranker": h.chunk_id not in rrf_order,
        }
        for h in selected
    ]
)

In [ ]:
# The grounded answer, with every claim traceable to a filing.
from fmis.rag.answer import ensure_model_available, generate_answer

ensure_model_available()
grounded = generate_answer(QUESTION, selected)

print(grounded.answer)
print("\n" + "=" * 78 + "\nSOURCES\n")
for citation in grounded.citations:
    print(f"{citation.marker} {citation.citation_label}")
    print(f"     file    : {citation.source_file}")
    print(f"     excerpt : {citation.excerpt[:180]}...\n")

print(f"warnings: {grounded.warnings or 'none'}")

In [ ]:
# The refusal path: a question the filings cannot answer must not be answered.
from fmis.rag.pipeline import answer_question

unanswerable = answer_question(
    "What is the CEO's home address and personal mobile phone number?"
)
print(unanswerable["answer"])
print(f"\nrefused  : {unanswerable['refused']}")
print(f"citations: {len(unanswerable['citations'])}")

In [ ]:
# Full demonstration set, written to evidence/runs/rag_answers.md
run("rag-demo")

answers = evidence("rag_answers.json")
answers["summary"]

---
## 5 · Lineage — START / COMPLETE / FAIL per stage

Every stage above ran inside an OpenLineage context manager. Note the `FAIL` event from the
tainted quality-gate run: the failure is recorded in lineage *and* propagated to halt the
pipeline.

In [ ]:
events = [
    json.loads(line)
    for line in settings.openlineage_file.read_text().splitlines()
    if line.strip()
]

frame = pd.DataFrame(
    [
        {
            "job": e["job"]["name"],
            "state": e["eventType"],
            "inputs": len(e.get("inputs") or []),
            "outputs": len(e.get("outputs") or []),
        }
        for e in events
    ]
)

print(f"{len(events)} events emitted\n")
frame.pivot_table(index="job", columns="state", values="inputs", aggfunc="count").fillna(0).astype(int)

In [ ]:
# The FAIL event, with the error facet carrying the reason.
for event in events:
    if event["eventType"] == "FAIL":
        facet = (event["run"].get("facets") or {}).get("errorMessage", {})
        print(f"job    : {event['job']['name']}")
        print(f"message: {facet.get('message', '')[:400]}\n")

---
## Orchestration

The DAG cannot be triggered from this kernel (Airflow lives in its own virtualenv), so run
these in a terminal and screenshot the graph view:

```bash
make airflow-init
make airflow-up                  # UI on :8080, admin / admin
make airflow-trigger             # production pipeline: all green
make airflow-trigger-failure     # gate fails, gold_merge SKIPPED
```

The second run is the one that matters for the rubric: `quality_gate_tainted` is **failed**
and `gold_merge_must_not_run` is **skipped** — Airflow never executed it, because the
default `all_success` trigger rule means a task downstream of a failure is not scheduled.

Save the screenshots to `evidence/screenshots/`.

---
## Summary

In [ ]:
print(f"""
1  INGESTION
   messages produced       : {scorecard['messages_produced']:,}
   admitted to lakehouse   : {scorecard['messages_accepted']:,}
   quarantined + DLQ       : {scorecard['messages_rejected']:,}
   injected faults escaped : {scorecard['faults_escaped_total']}

2  LAKEHOUSE
   bronze rows             : {bronze.count():,}
   silver rows             : {silver.count():,}
   gold rows (per ticker)  : {gold.count():,}
   bad writes refused      : {enforcement['attempts_passed']}/{enforcement['attempts_total']}

3  RAG
   filings indexed         : {index_stats['filings']}
   chunks                  : {index_stats['chunks']:,}
   questions answered      : {answers['summary']['answered']}/{answers['summary']['questions']}
   answers with citations  : {answers['summary']['answers_with_citations']}

4  QUALITY GATE
   clean run               : {gate['successful']}/{gate['evaluated']} passed
   tainted run             : failed as designed, pipeline halted

5  LINEAGE
   events emitted          : {len(events)}
""")

spark.stop()